In [ ]:
# ============================================================
#  DEDUPLICATION v3.1 — OpenAlex-only, oa_* schema
#
#  Patched from v2. Changes tagged  # [FIX]  (v3) and  # [OA]  (this pass).
#  Pure dedup only — NO relevance screening (run add_relevance() from
#  tick_dedup_screen.py AFTER, on the canonical set).
#
#  OpenAlex-only tuning ([OA]):
#   - journal kept AS-IS: no stopword list, EXACT match only (no fuzzy).
#     OpenAlex emits one canonical display_name per journal, so this is safe.
#   - fuzzy applies to TITLE only; journal & author are exact confirmers.
#   - TYPE_PRIORITY rebuilt on the real 19-value OpenAlex type vocabulary
#     (verified against developers.openalex.org/api-reference/work-types).
#   - optional PROFILE block prints columns / dtypes / type counts / null rates.
#
#  Fixes vs v2:
#   [FIX-1] author_fp now splits on ';' OR '|' (harvester joins authors with ' | ')
#   [FIX-2] schema wired to oa_* (oa_type/oa_abstract/... ) + source_database tag
#   [FIX-3] Stage 4 requires shared author at FUZZY_THRESHOLD; journal-only
#           merges allowed only at a STRICTER threshold (closes precision hole)
#   [FIX-4] Stage 4 blocks on a content token (leading articles stripped) to
#           avoid giant "a/an/the" blocks and O(block^2) blow-up
#   [FIX-5] year tolerance defaulted to a precision-safe value (was 3)
#   [FIX-6] type_pri tolerates hyphenated OpenAlex types (book-chapter, etc.)
#
#  Input : data_raw/combined.parquet   (openalex_harvester oa_* output)
#  Output: master_dedupe_audit.xlsx, master_canonical.xlsx, qc/*.xlsx
# ============================================================

## ---- 1. setup ----
from google.colab import drive
drive.mount('/content/drive')
!pip -q install rapidfuzz openpyxl pandas

import os, re, time, unicodedata
import pandas as pd, numpy as np
from rapidfuzz import fuzz
pd.set_option('display.max_columns', None)

## ---- 2. config ----
FOLDER = "/content/drive/MyDrive/COGNOSX/ticks"  # <-- EDIT
IN_PATH  = os.path.join(FOLDER, "data_raw/combined.parquet")
OUT_DIR  = os.path.join(FOLDER, "preprocessing_data")
QC_DIR   = os.path.join(OUT_DIR, "qc")
os.makedirs(QC_DIR, exist_ok=True)
AUDIT_PATH = os.path.join(OUT_DIR, "master_dedupe_audit.xlsx")
CANON_PATH = os.path.join(OUT_DIR, "master_canonical.xlsx")

# tunable parameters
PROFILE               = True  # [OA] print a one-shot profile of the input file
FUZZY_THRESHOLD       = 90    # token_sort_ratio on TITLE only  [OA]
FUZZY_YEAR_TOLERANCE  = 1     # [FIX-5] was 3; 1 is precision-safe. Raise to 2
                              #         for preprint-heavy corpora (pre/post gap).
MIN_TITLE_TOKENS      = 3
MIN_TITLE_CHARS       = 15
AUTHOR_FP_MODE        = "last_initial"   # tokenbag | last_initial | last

## ---- 3. load ----
df = pd.read_parquet(IN_PATH).fillna("").reset_index(drop=True)
print("Loaded:", len(df), "rows,", len(df.columns), "columns")
if "record_id" not in df.columns:
    df["record_id"] = [f"REC-{i+1:06d}" for i in range(len(df))]
# [FIX-2] single OpenAlex pull has no source column; tag it so the cross-DB
#         report/groupbys work. When you feed the MERGED 7-source master, drop
#         this line and make sure each source is already tagged on ingest.
if "source_database" not in df.columns:
    df["source_database"] = "openalex"

## ---- 3.5 optional profile (verify names/types before deduping) ----  [OA]
if PROFILE:
    print("\n=== INPUT PROFILE ===")
    print("columns:", list(df.columns))
    key = ["oa_id","oa_doi","oa_pmid","oa_title","oa_abstract","oa_authors",
           "oa_year","oa_journal","oa_type","oa_keywords"]
    present = [c for c in key if c in df.columns]
    missing = [c for c in key if c not in df.columns]
    if missing: print("!! MISSING expected columns:", missing)
    empty_rate = ((df[present].astype(str).apply(lambda s: s.str.strip())
                   .isin(["", "none", "nan", "null", "[]"]))).mean().round(3)
    print("\nempty/NA rate per key column:\n" + empty_rate.to_string())
    if "oa_type" in df.columns:
        print("\noa_type value_counts:\n" + df["oa_type"].value_counts().head(20).to_string())
    if "oa_authors" in df.columns:
        s = df.loc[df["oa_authors"].astype(str).str.len() > 0, "oa_authors"].head(1)
        if len(s): print("\nauthors delimiter sample:", repr(s.iloc[0][:120]))
    print("=== END PROFILE ===\n")

## ---- 4. normalization helpers (cached once) ----
_WS = re.compile(r"\s+"); _PUNCT = re.compile(r"[^\w\s]", re.UNICODE)
NA_LIKE = {"", "none", "null", "nan", "n/a", "[]", "{}", "na"}

def strip_acc(x):
    return "".join(c for c in unicodedata.normalize("NFKD", x) if not unicodedata.combining(c))

def clean_title(s):
    s = strip_acc(str(s or "").lower().strip())
    if s in NA_LIKE: return ""
    s = _PUNCT.sub(" ", s); s = _WS.sub(" ", s).strip()
    return s

def norm_doi(d):
    d = str(d or "").strip().lower()
    for p in ("https://doi.org/","http://dx.doi.org/","https://dx.doi.org/","doi:"):
        d = d.replace(p, "")
    d = d.strip()
    return d if d.startswith("10.") else ""

def norm_year(y):
    m = re.search(r"(19|20)\d{2}", str(y or "")); return m.group(0) if m else ""

# [FIX-4] block key = first CONTENT token (leading articles stripped) so
#         "A ...", "An ...", "The ..." don't collapse into three mega-blocks.
LEAD_STOP = {"a", "an", "the"}
def block_token(title_clean):
    toks = title_clean.split()
    i = 0
    while i < len(toks) and toks[i] in LEAD_STOP:
        i += 1
    return toks[i] if i < len(toks) else ""

# [OA] journal kept AS-IS. OpenAlex emits one canonical display_name per journal,
#      so we only case/space-fold for a robust EXACT match — no stopword list,
#      no accent/punctuation stripping, no fuzzy. Raw oa_journal stays in output.
JOURNAL_NA = {"", "none", "null", "nan", "n/a", "na"}
def journal_key(s):
    s = str(s or "").strip().lower()
    if s in JOURNAL_NA: return ""
    return _WS.sub(" ", s)

def journal_same(a, b):          # [OA] exact match (never fuzzy)
    return bool(a) and a == b

# --- risky / generic title blocklist ---
GENERIC_TITLES = frozenset({
    "front matter","back matter","summary","summaries","reviews in brief","publications received",
    "editorial","introduction","preface","contents","table of contents","index","book review",
    "letter","news","announcement","abstract","poster","supplement","no title","retracted",
    "bibliographie","corrigendum","erratum","obituary","in this issue","highlights",
    "acknowledgements","author index","subject index","calendar","meeting","conference","abstracts",
})
def is_risky(t):
    if not t: return True
    if t in GENERIC_TITLES: return True
    if len(t.split()) < MIN_TITLE_TOKENS: return True
    if len(t) < MIN_TITLE_CHARS: return True
    return False

# --- author fingerprint ---
def author_fp(authors_flat, mode=AUTHOR_FP_MODE):
    a = str(authors_flat or "").strip()
    if a.lower() in NA_LIKE: return ""
    out = []
    for au in re.split(r"[;|]", a):                       # [FIX-1] split on ';' OR '|'
        au = strip_acc(au.lower().strip())
        if not au or au in NA_LIKE: continue
        if "," in au:
            left, right = au.split(",", 1)
            left  = _WS.sub(" ", _PUNCT.sub(" ", left)).strip()
            right = _WS.sub(" ", _PUNCT.sub(" ", right)).strip()
            lt = [t for t in left.split() if t]; rt = [t for t in right.split() if t]
            if not lt: continue
            if mode == "last":
                out.append(lt[0])
            else:
                if not rt: continue
                if mode == "tokenbag":
                    out.append("_".join(sorted(lt + rt)))
                else:  # last_initial
                    out.append(f"{lt[0]}|{rt[0][:1]}")
        else:
            au = _WS.sub(" ", _PUNCT.sub(" ", au)).strip(); toks = [t for t in au.split() if t]
            if len(toks) < 2:
                if mode == "last" and toks: out.append(toks[-1])
                continue
            if mode == "last":
                out.append(toks[-1])
            elif mode == "tokenbag":
                out.append("_".join(sorted(toks)))
            else:  # last_initial  ("First Last" -> "last|f")
                out.append(f"{toks[-1]}|{toks[0][:1]}")
    return ";".join(sorted(set(out)))

def fp_overlap(a, b):
    if not a or not b: return 0
    return len(set(a.split(";")) & set(b.split(";")))

t0 = time.perf_counter()
df["title_clean"] = df["oa_title"].map(clean_title)
df.loc[df["title_clean"].map(is_risky), "title_clean"] = ""
df["doi_norm"]    = df["oa_doi"].map(norm_doi)
df["year_norm"]   = df["oa_year"].map(norm_year)
df["authors_fp"]  = df["oa_authors"].map(author_fp)
df["journal_key"] = df["oa_journal"].map(journal_key)          # [OA] exact key
df["block_tok"]   = df["title_clean"].map(block_token)          # [FIX-4]
print(f"Normalization: {time.perf_counter()-t0:.1f}s | risky titles blanked: {(df['title_clean']=='').sum()}")

## ---- 5. union-find with cluster-level DOI tracking ----
parent = {}; cluster_dois = {}
def find(x):
    parent.setdefault(x, x); r = x
    while parent[r] != r: r = parent[r]
    while parent[x] != r: parent[x], x = r, parent[x]
    return r
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb
        cluster_dois.setdefault(rb, set()).update(cluster_dois.get(ra, set()))
def doi_conflict(a, b):
    da = cluster_dois.get(find(a), set()); db = cluster_dois.get(find(b), set())
    return bool(da and db and da != db and not (da & db))

for i in df.index:
    find(i)
    d = df.at[i, "doi_norm"]
    if d: cluster_dois[i] = {d}

M = {}

## Stage 1: exact DOI (no author gate)
n = 0
for doi, g in df[df["doi_norm"].ne("")].groupby("doi_norm"):
    ix = g.index.tolist()
    for j in ix[1:]: union(ix[0], j); n += 1
M["1_doi_exact"] = n

## Stage 2a: title_clean + authors_fp + year (exact, author-gated)
n = 0
el = df["title_clean"].ne("") & df["authors_fp"].ne("") & df["year_norm"].ne("")
key = df["title_clean"] + "||" + df["authors_fp"] + "||" + df["year_norm"]
for k, g in df[el].groupby(key[el]):
    ix = g.index.tolist()
    for j in ix[1:]:
        if find(ix[0]) != find(j) and not doi_conflict(ix[0], j):
            union(ix[0], j); n += 1
M["2a_title_fp_year"] = n

## Stage 2b: title_clean + journal_key + year (exact)
n = 0
el = df["title_clean"].ne("") & df["journal_key"].ne("") & df["year_norm"].ne("")
key = df["title_clean"] + "||" + df["journal_key"] + "||" + df["year_norm"]
for k, g in df[el].groupby(key[el]):
    ix = g.index.tolist()
    for j in ix[1:]:
        if find(ix[0]) != find(j) and not doi_conflict(ix[0], j):
            union(ix[0], j); n += 1
M["2b_title_journal_year"] = n

## Stage 3: title_clean + year, gated by >=1 shared author
n = 0
el = df["title_clean"].ne("") & df["year_norm"].ne("")
key = df["title_clean"] + "||" + df["year_norm"]
for k, g in df[el].groupby(key[el]):
    ix = g.index.tolist()
    if len(ix) < 2: continue
    for a in range(len(ix)):
        for b in range(a + 1, len(ix)):
            ia, ib = ix[a], ix[b]
            if find(ia) == find(ib) or doi_conflict(ia, ib): continue
            fa, fb = df.at[ia, "authors_fp"], df.at[ib, "authors_fp"]
            if fa and fb and fp_overlap(fa, fb) < 1: continue
            union(ia, ib); n += 1
M["3_title_year_authorgate"] = n

## Stage 4: FUZZY on TITLE only, gated by EXACT shared-author OR EXACT journal.
##          [OA] journal & author are exact confirmers; only the title is fuzzy.
##          [FIX-4] block on content token; report max block size.
##          To require author evidence for every fuzzy merge, delete `or same_journal`.
n = 0
cand = df[df["title_clean"].ne("")]
max_block = 0
for tok, block in cand.groupby("block_tok"):
    if tok == "": continue
    ix = block.index.tolist()
    max_block = max(max_block, len(ix))
    if len(ix) < 2: continue
    info = [(i, df.at[i,"title_clean"], df.at[i,"year_norm"], df.at[i,"authors_fp"], df.at[i,"journal_key"]) for i in ix]
    for a in range(len(info)):
        ia, ta, ya, fa, ja = info[a]
        for b in range(a + 1, len(info)):
            ib, tb, yb, fb, jb = info[b]
            if find(ia) == find(ib) or doi_conflict(ia, ib): continue
            if ya and yb and abs(int(ya) - int(yb)) > FUZZY_YEAR_TOLERANCE: continue
            if abs(len(ta) - len(tb)) > 25: continue
            shared_author = bool(fa and fb and fp_overlap(fa, fb) >= 1)   # exact
            same_journal  = journal_same(ja, jb)                          # [OA] exact
            if not (shared_author or same_journal):
                continue
            if fuzz.token_sort_ratio(ta, tb) >= FUZZY_THRESHOLD:          # [OA] title only
                union(ia, ib); n += 1
M["4_fuzzy_title_gated"] = n
print(f"Stage 4 largest block: {max_block} records")           # watch for runaway blocks

## ---- 6. assign study_id, canonical, provenance ----
df["cluster"] = [find(i) for i in df.index]

# [OA] real OpenAlex 'type' vocabulary (19 values, one per work) — exact lookup.
#      Verified: developers.openalex.org/api-reference/work-types
TYPE_PRIORITY = {
    "article": 10, "review": 8, "book-chapter": 5, "book": 5, "report": 4,
    "reference-entry": 3, "dataset": 3, "preprint": 3, "dissertation": 2,
    "standard": 2, "editorial": 1, "letter": 1, "erratum": 1, "retraction": 1,
    "peer-review": 1, "supplementary-materials": 1, "libguides": 1,
    "paratext": 0, "other": 0,
}
def type_pri(t):
    return TYPE_PRIORITY.get(str(t or "").strip().lower(), 0)   # [OA] exact

# [FIX-2] canonical scoring uses oa_* columns
CORE = ["oa_title","oa_authors","oa_year","oa_doi","oa_journal","oa_abstract","oa_keywords","oa_type"]
df["_meta"] = sum((df[c].astype(str).str.len() > 0).astype(int) for c in CORE if c in df.columns)
df["_score"] = (df["doi_norm"].ne("").astype(int)*100
                + df["oa_type"].map(type_pri)*3                 # [FIX-2] was document_type
                + df["_meta"]*5
                + (df["oa_abstract"].astype(str).str.len() > 0).astype(int)*6)  # [FIX-2] was abstract
canon_idx = df.sort_values(["_score","record_id"], ascending=[False,True]).groupby("cluster").head(1).index
df["is_canonical"] = df.index.isin(canon_idx)

order = {c: n for n, c in enumerate(df["cluster"].tolist())}
uniq  = sorted(df["cluster"].unique(), key=lambda c: order[c])
cl2stu = {c: f"STU-{i+1:06d}" for i, c in enumerate(uniq)}
df["study_id"] = df["cluster"].map(cl2stu)
found = df.groupby("study_id")["source_database"].apply(lambda s: "; ".join(sorted(set(s))))
df["found_in"]  = df["study_id"].map(found)
df["n_sources"] = df.groupby("study_id")["source_database"].transform("nunique")
df["n_records"] = df.groupby("study_id")["record_id"].transform("count")

## ---- 7. report ----
nr, ns = len(df), df["study_id"].nunique()
print("\n=== STAGE MERGES ===")
for k, v in M.items(): print(f"  {k}: {v}")
print("\n=== PRISMA identification ===")
print(f"Records identified: {nr}")
print(df["source_database"].value_counts().to_string())
print(f"\nDuplicates removed:        {nr - ns}")
print(f"Records after de-dup:      {ns}")
print("\nStudies by #databases:")
print(df.drop_duplicates('study_id')['n_sources'].value_counts().sort_index().to_string())
print(f"\nLargest study: {df.groupby('study_id').size().max()} records")

## ---- 8. QC files ----
study_has_doi = df.groupby("study_id")["doi_norm"].transform(lambda s: s.ne("").any())
qc_canon_no_doi = df[df["is_canonical"] & df["doi_norm"].eq("") & study_has_doi]
tt = df.groupby("study_id")["title_clean"].nunique()
qc_multi_title  = df[df["study_id"].isin(tt[tt > 1].index)]
qc_big          = df[df["n_records"] >= 8]
print(f"\n=== QC ===")
print(f"Canonical missing DOI while dup has one: {len(qc_canon_no_doi)}")
print(f"Rows in multi-distinct-title studies:    {len(qc_multi_title)}")
print(f"Rows in large studies (>=8 records):     {len(qc_big)}")

## ---- 9. save ----
def safe_xlsx(frame, path, sheet):
    # [OA] robust XML cleaner: C0 (minus tab/LF/CR) + DEL + C1 controls +
    #      Unicode non-characters + lone surrogates. OpenAlex multilingual text
    #      carries these (esp. C1 from Latin-1/Win-1252 mis-decoding); lxml rejects them.
    ILLEGAL = re.compile(
        r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f\ufdd0-\ufdef\ufffe\uffff\ud800-\udfff]"
    )
    out = frame.copy()
    for c in out.columns:
        out[c] = out[c].map(lambda v: ILLEGAL.sub(" ", str(v))[:32000])
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        out.to_excel(xw, sheet_name=sheet, index=False)

# [FIX-2] front columns wired to oa_* names for a readable audit sheet
front = ["study_id","record_id","is_canonical","n_sources","n_records","found_in",
         "source_database","oa_id","oa_title","oa_authors","oa_year","oa_doi","oa_journal",
         "oa_abstract","oa_keywords","oa_pmid","oa_type","oa_funders","oa_cited_by_count","oa_topic"]
front = [c for c in front if c in df.columns]
helper = ["title_clean","doi_norm","year_norm","authors_fp","block_tok",
          "journal_key","cluster","_meta","_score"]
rest = [c for c in df.columns if c not in front + helper]
audit = df[front + rest]

# [OA] .fillna("") at load turned numeric cols (oa_fwci, oa_cited_by_count, ...)
#      into object/str with "" for missing, which pyarrow can't write. This
#      restores null + numeric dtypes per column before writing parquet.
def to_parquet_safe(frame, path):
    out = frame.copy()
    for c in out.columns:
        s = out[c]
        if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_bool_dtype(s):
            continue
        s = s.replace("", pd.NA)
        conv = pd.to_numeric(s, errors="coerce")
        if s.notna().sum() > 0 and conv.notna().sum() == s.notna().sum():
            out[c] = conv                       # every non-null value was numeric
        else:
            out[c] = s.astype("string")
    out.to_parquet(path, index=False)

# full-fidelity parquet (fast, DuckDB-friendly). The 334k-row xlsx below is
# slow/heavy — prefer parquet programmatically; keep xlsx for human QC files.
to_parquet_safe(audit, os.path.join(OUT_DIR, "master_dedupe_audit.parquet"))
to_parquet_safe(audit[df["is_canonical"]], os.path.join(OUT_DIR, "master_canonical.parquet"))
safe_xlsx(audit, AUDIT_PATH, "dedupe_audit")
safe_xlsx(audit[df["is_canonical"]], CANON_PATH, "canonical")
safe_xlsx(qc_canon_no_doi[front], os.path.join(QC_DIR,"qc_canonical_missing_doi.xlsx"), "qc")
safe_xlsx(qc_multi_title[front],  os.path.join(QC_DIR,"qc_multi_title_studies.xlsx"), "qc")
safe_xlsx(qc_big[front],          os.path.join(QC_DIR,"qc_large_studies.xlsx"), "qc")

print("\nSaved:")
print(" ", AUDIT_PATH, f"({len(audit)} rows)")
print(" ", CANON_PATH, f"({df['is_canonical'].sum()} unique studies)")
print(" ", QC_DIR, "(3 review files)")

## ---- 10. NEXT: screening (kept separate) ----
# from tick_dedup_screen import add_relevance, screening_report
# canon = df[df["is_canonical"]].copy()
# canon = add_relevance(canon)
# print(screening_report(canon))
# canon.to_parquet(os.path.join(OUT_DIR, "canonical_screened.parquet"), index=False)